# RAG Evaluation

Evaluate a conversational RAG pipeline on a curated dataset using RAGAS metrics.

The pipeline:
1. Chunk a document and index it into FAISS
2. Build a conversational retrieval chain (question rewriter → retriever → LLM)
3. Run each question through the chain and collect `(response, retrieved_contexts)`
4. Score with RAGAS: Context Recall, Context Precision, Faithfulness, Answer Relevancy, Factual Correctness, Semantic Similarity
5. Compute MRR independently from retrieved chunk rankings

## Setup

In [ ]:
import os
import json
import uuid
from pathlib import Path
from operator import itemgetter

from dotenv import load_dotenv
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
HF_TOKEN       = os.getenv("HF_TOKEN")

ROOT       = Path("../")
DOCUMENT   = ROOT / "experiments" / "data" / "doc_11_which_gpu_for_deep_learning.txt"
DATASET    = ROOT / "experiments" / "dataset.json"
FAISS_BASE = ROOT / "experiments" / "faiss_index"

## Models

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEndpointEmbeddings

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

def get_llm():
    return ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        google_api_key=GOOGLE_API_KEY,
        temperature=0.5,
        max_output_tokens=512,
    )

def get_embeddings():
    return HuggingFaceEndpointEmbeddings(
        huggingfacehub_api_token=HF_TOKEN,
        model=EMBEDDING_MODEL,
    )

## Dataset

In [ ]:
with open(DATASET) as f:
    dataset = json.load(f)

print(f"{len(dataset)} examples")
for ex in dataset:
    print(f"  [{ex['question_type']}] {ex['question']}")

## Indexing

Chunk the document and build a FAISS index.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

def build_index(chunk_size=1000, chunk_overlap=200):
    session_id = f"nb_{uuid.uuid4().hex[:8]}"
    index_path = FAISS_BASE / session_id
    FAISS_BASE.mkdir(parents=True, exist_ok=True)

    loader = TextLoader(str(DOCUMENT), encoding="utf-8")
    docs = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    chunks = splitter.split_documents(docs)
    print(f"Chunks: {len(chunks)}  (size={chunk_size}, overlap={chunk_overlap})")

    embeddings = get_embeddings()
    vectorstore = FAISS.from_documents(chunks, embeddings)
    vectorstore.save_local(str(index_path))
    print(f"Index saved → {index_path}")

    return str(index_path)

index_path = build_index(chunk_size=1000, chunk_overlap=200)

## RAG Chain

Two-step LCEL chain: rewrite the question as standalone → retrieve → answer.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Given the conversation history and the latest user question, rewrite it as a "
     "standalone question. Do not answer — only reformulate if needed, else return as-is."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

qa_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer using only the retrieved context below. "
     "If the answer is not in the context, say 'I don't know.' "
     "Keep your answer under three sentences.\n\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

In [ ]:
def build_chain(index_path, k=5, search_type="similarity", fetch_k=20, lambda_mult=0.5):
    embeddings = get_embeddings()
    vectorstore = FAISS.load_local(
        index_path, embeddings, allow_dangerous_deserialization=True
    )

    search_kwargs = {"k": k}
    if search_type == "mmr":
        search_kwargs.update({"fetch_k": fetch_k, "lambda_mult": lambda_mult})

    retriever = vectorstore.as_retriever(
        search_type=search_type, search_kwargs=search_kwargs
    )

    retrieved_docs = []

    def format_docs(docs):
        retrieved_docs.clear()
        retrieved_docs.extend(docs)
        return "\n\n".join(d.page_content for d in docs)

    llm = get_llm()

    question_rewriter = (
        {"input": itemgetter("input"), "chat_history": itemgetter("chat_history")}
        | contextualize_prompt
        | llm
        | StrOutputParser()
    )

    chain = (
        {
            "context": question_rewriter | retriever | format_docs,
            "input": itemgetter("input"),
            "chat_history": itemgetter("chat_history"),
        }
        | qa_prompt
        | llm
        | StrOutputParser()
    )

    return chain, retrieved_docs


def ask(chain, retrieved_docs, question, chat_history=None):
    answer = chain.invoke({"input": question, "chat_history": chat_history or []})
    contexts = [d.page_content for d in retrieved_docs]
    return {"answer": answer, "contexts": contexts}

### Quick smoke test

In [ ]:
chain, retrieved_docs = build_chain(index_path)
result = ask(chain, retrieved_docs, "How much faster is the Tesla A100 compared to the Tesla V100?")

print("Answer:", result["answer"])
print(f"\nContexts retrieved: {len(result['contexts'])}")
print("\nFirst context snippet:")
print(result["contexts"][0][:300])

## Build RAGAS Samples

Run every dataset question through the chain and collect `SingleTurnSample` objects.

In [ ]:
from ragas import EvaluationDataset, SingleTurnSample

chain, retrieved_docs = build_chain(index_path)
samples      = []
samples_meta = []  # kept separately for MRR computation

for ex in dataset:
    result = ask(chain, retrieved_docs, ex["question"])
    samples.append(
        SingleTurnSample(
            user_input=ex["question"],
            response=result["answer"],
            retrieved_contexts=result["contexts"],
            reference=ex["answer"],
        )
    )
    samples_meta.append({
        "reference":          ex["answer"],
        "retrieved_contexts": result["contexts"],
    })
    print(f"✓ {ex['question'][:70]}")

ragas_dataset = EvaluationDataset(samples=samples)
print(f"\n{len(samples)} samples ready")

## MRR

Computed independently from RAGAS. For each question, find the rank of the first retrieved chunk that contains at least 2 keywords from the reference answer.

In [ ]:
import math

def compute_mrr(samples_meta):
    NO_ANSWER = "i don't know."
    mrr_scores = []
    for meta in samples_meta:
        reference = meta["reference"].strip()
        if reference.lower() == NO_ANSWER:
            mrr_scores.append(float("nan"))
            continue
        ref_words = {w.lower() for w in reference.split() if len(w) > 4}
        hit_rank  = None
        for rank, chunk in enumerate(meta["retrieved_contexts"], start=1):
            if sum(1 for w in ref_words if w in chunk.lower()) >= 2:
                hit_rank = rank
                break
        mrr_scores.append(1.0 / hit_rank if hit_rank else 0.0)
    return mrr_scores

## Evaluate

Score with RAGAS metrics using Gemini as the judge LLM, then append MRR.

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    LLMContextRecall,
    LLMContextPrecisionWithReference,
    Faithfulness,
    AnswerRelevancy,
    FactualCorrectness,
    SemanticSimilarity,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

judge_llm        = LangchainLLMWrapper(get_llm())
judge_embeddings = LangchainEmbeddingsWrapper(get_embeddings())

result = evaluate(
    dataset=ragas_dataset,
    metrics=[
        LLMContextRecall(),
        LLMContextPrecisionWithReference(),
        Faithfulness(),
        AnswerRelevancy(),
        FactualCorrectness(),
        SemanticSimilarity(),
    ],
    llm=judge_llm,
    embeddings=judge_embeddings,
)

results_df         = result.to_pandas()
results_df["mrr"]  = compute_mrr(samples_meta)

scores = results_df.mean(numeric_only=True).to_dict()
print("\nAggregate scores:")
for metric, score in scores.items():
    print(f"  {metric:<45} {score:.4f}")

## Results

In [ ]:
import pandas as pd

display_cols = [
    "user_input",
    "response",
    "reference",
    "context_recall",
    "llm_context_precision_with_reference",
    "faithfulness",
    "answer_relevancy",
    "factual_correctness(mode=f1)",
    "semantic_similarity",
    "mrr",
]

pd.set_option("display.max_colwidth", 80)
results_df[[c for c in display_cols if c in results_df.columns]]